# 02e — Combine all feature tables

Create one analysis table with **one row per `SUBJECT`/`SPELL_IDENTIFIER`**. No surrogate `episode_row_id` is used. If duplicate spell identifiers exist, this notebook keeps the first row in each input table.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema

DATA_DIR = get_path('data/synthetic/10000p')
DATA_DIR = get_data_root_from_schema() 
DATA_PROC_DIR = DATA_DIR / 'processed'

KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

def keep_unique_rows(df: pd.DataFrame, keys: list[str], name: str) -> pd.DataFrame:
    """Keep the first source row for the given granularity keys."""
    missing = [c for c in keys if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing key columns: {missing}")
    result = df.copy()
    before = len(result)
    result = result.drop_duplicates(keys, keep="first").reset_index(drop=True)
    print(f"{name} ({'-'.join(keys)}): {before:,} rows -> {len(result):,} unique records")
    return result

print(f"Using data from: {DATA_DIR}")


Using data from: /app/data/synthetic/10000p


## Load the base episodes and derived features

In [8]:
# Load episodes
df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")
df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])
print(df_episodes.dtypes)

# 1. Load base episodes
df_base = keep_unique_rows(
    df_episodes,
    keys=["SUBJECT", "SPELL_IDENTIFIER"],
    name="base episodes",
)

# 2. Define feature files with their respective join keys
feature_sources = {
    # Spell-level features
    "phenotypes": {
        "path": DATA_PROC_DIR / "02a_derived_phenotypes_aggregated.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "history": {
        "path": DATA_PROC_DIR / "02b_derived_features_phenotypes.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "encounters": {
        "path": DATA_PROC_DIR / "02c_derived_features_encounters.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "medications": {
        "path": DATA_PROC_DIR / "02d_derived_features_medications.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "microbiology": {
        "path": DATA_PROC_DIR / "02e_derived_features_microbiology.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    # Patient-level features (broadcasts across spells)
    "demographics": {
        "path": DATA_DIR / "icare_demographic_anon.parquet",
        "keys": ["SUBJECT"]
    },
}

# 3. Validate existence and load features dynamically
loaded_features = {}
missing = []
for name, meta in feature_sources.items():
    path = meta["path"]
    if not path.exists():
        missing.append(name)
        continue

    # Read parquet or csv automatically based on extension
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    # Standardize keys to string
    for k in meta["keys"]:
        df[k] = df[k].astype(str)
    # Create dictionary
    loaded_features[name] = {
        "df": keep_unique_rows(df, meta["keys"], name),
        "keys": meta["keys"]
    }

if missing:
    raise FileNotFoundError(f"Missing feature files: {', '.join(missing)}. Ensure they are generated first.")


SUBJECT                                          int64
SPELL_IDENTIFIER                                 int64
ENCNTR_ID                                        int64
EPISODE_IDENTIFIER                               int64
ADMISSION_DATE                          datetime64[ns]
ADMISSION_TIME                          string[python]
AGE_AT_ADMISSION                                 int64
DISCHARGE_DATE                          datetime64[ns]
MAIN_SPECIALTY_CODE                              int64
MAIN_SPECIALTY_CODE_DESC                string[python]
INDEX_OF_MULTIPLE_DEPRIVATION_DECILE    string[python]
dtype: object
base episodes (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
phenotypes (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
history (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
encounters (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
medications (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records


## Merge safely

In [9]:
df_combined = df_base.copy()
df_combined.SUBJECT = df_combined.SUBJECT.astype(str)
df_combined.SPELL_IDENTIFIER = df_combined.SPELL_IDENTIFIER.astype(str)

for name, item in loaded_features.items():
    feature_df = item["df"]
    keys = item["keys"]

    # Drop overlapping columns (except join keys) to avoid duplicate suffixes
    overlapping = [
        c for c in feature_df.columns
        if c in df_combined.columns and c not in KEYS
    ]
    feature_df = feature_df.drop(columns=overlapping)

    # Toggle validation rule based on table grain (spell vs patient)
    validate_rule = "one_to_one" if len(keys) > 1 else "many_to_one"

    df_combined = df_combined.merge(
        feature_df,
        on=keys,
        how="left",
        validate=validate_rule,
    )
    print(f"Merged {name} via {keys}: shape {df_combined.shape}")

assert not df_combined.duplicated(["SUBJECT", "SPELL_IDENTIFIER"]).any(), "Duplicate spells detected post-merge!"
print(f"\nFinal combined dataset ready: {len(df_combined):,} rows, {len(df_combined.columns):,} columns.")


Merged phenotypes via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 84)
Merged history via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 174)
Merged encounters via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 176)
Merged medications via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 177)
Merged microbiology via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 181)
Merged demographics via ['SUBJECT']: shape (10000, 188)

Final combined dataset ready: 10,000 rows, 188 columns.


In [10]:
filename = "03_derived_analysis_dataset.parquet"
output_path = DATA_PROC_DIR / filename
df_combined.to_parquet(output_path, index=False)
print(f"Saved {len(df_combined):,} rows and {len(df_combined.columns):,} columns to {output_path}")
display(df_combined.head())
print(df_combined.columns.tolist())


Saved 10,000 rows and 188 columns to /app/data/synthetic/10000p/processed/03_derived_analysis_dataset.parquet


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,time_to_culture_days,time_to_culture_hours,esbl_status,BIRTH_DATE,DEATH_DATE,ETHNICITY,ETHNICITY_DES,GENDER,GENDER_DESC,UPDATE_DT_TM
0,10001,7867287,2998828,3749211,2023-02-12,00:52:00,64,2023-02-26,140,oral surgery,...,NaN,NaN,unknown,1958-08-07,2021-01-03,L,Any other Asian background,M,Male,2023-01-05
1,10002,6551735,8909338,9353593,2022-11-16,23:43:00,72,2022-12-01,320,cardiology,...,9.0,216.0,unknown,1955-02-09,2023-11-15,N,Black or Black British - African,U,Unknown,2023-07-03
2,10003,4441228,1307783,3518105,2022-05-12,20:10:00,29,2022-05-27,320,cardiology,...,12.0,292.0,unknown,1963-03-27,2025-10-17,A,White - British,U,Unknown,2023-10-23
3,10004,5593202,5695692,6593442,2023-10-11,00:52:00,19,2023-10-30,300,general medicine,...,NaN,NaN,unknown,1976-04-02,2025-10-28,M,Black or Black British - Caribbean,I,Indeterminate,2023-10-10
4,10005,6532182,7455884,6203572,2023-08-14,05:42:00,50,2023-08-18,300,general medicine,...,NaN,NaN,unknown,1986-04-04,2024-10-18,K,Asian or Asian British - Bangladeshi,I,Indeterminate,2023-01-06


['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarello_hx_recent_hosp_90d', 'tumbarello_hx_recent_ab